In [1]:
import carla, random, pygame, numpy as np

pygame 2.6.1 (SDL 2.28.4, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# Core elements in carla are: actors, blueprints, worlds, and clients
client = carla.Client('localhost', 2000)

# Set the map to Town06. Silence output
try:
    client.load_world('Town06')
except RuntimeError:
    pass  # Ignore timeout or connection errors

In [3]:
# Get the world object
world = client.get_world()

# settings = world.get_settings()
# settings.synchronous_mode = True # Enables synchronous mode. THIS BREAKS THE SIM
# settings.fixed_delta_seconds = 0.01 # Set a variable time-step
# world.apply_settings(settings)

In [4]:
# Retrieve the spectator object
spectator = world.get_spectator()

# Update the spectator's location and rotation to the center lane of our scenario
# default_location = carla.Location(x=17.092312, y=244.485397, z=0.500000)
# default_rotation = carla.Rotation(pitch=360.000000, yaw=1.780838, roll=0.000000)

# Top-down view of our scenario
default_location = carla.Location(x=116, y=244.485397, z=100.000000) # At a height of z=100 meters, track length is about 200 meters
default_rotation = carla.Rotation(pitch=270.000000, yaw=270, roll=0.000000)

spectator.set_transform(carla.Transform(default_location, default_rotation))

In [ ]:
# Spawn our ego and NPC

ego_blueprint = world.get_blueprint_library().find('vehicle.dodge.charger_2020')
ego_spawn_location = carla.Location(x=21, y=244.485397, z=0.500000)
ego_spawn_rotation = carla.Rotation(pitch=0.000000, yaw=0, roll=0.000000)
ego_spawn_point = carla.Transform(ego_spawn_location, ego_spawn_rotation)
ego_blueprint.set_attribute('role_name', 'hero') # Set the role name of the ego vehicle

npc_blueprint = world.get_blueprint_library().find('vehicle.dodge.charger_police_2020')
npc_spawn_location = carla.Location(x=21, y=(244.485397 - 3.5), z=0.500000)
npc_spawn_rotation = carla.Rotation(pitch=0.000000, yaw=0, roll=0.000000)
npc_spawn_point = carla.Transform(npc_spawn_location, npc_spawn_rotation)

ego_vehicle = world.spawn_actor(ego_blueprint, ego_spawn_point)
npc_vehicle = world.spawn_actor(npc_blueprint, npc_spawn_point)

# Height and width of window
W, H = 758, 396

cam_bp = world.get_blueprint_library().find("sensor.camera.rgb")
cam_bp.set_attribute("image_size_x", str(W))
cam_bp.set_attribute("image_size_y", str(H))
cam_bp.set_attribute("fov", "135")  # Set FOV in degrees (default is 90, higher = wider view)

# Driver's point of view: positioned at driver's head, looking forward through windshield
camera = world.spawn_actor(cam_bp, carla.Transform(
    carla.Location(x=0.2, y=-0.36, z=1.2),  # Driver's head position (x=0.5 for front/back, y=-0.36 for left/right, z=1.3 for head height)
    carla.Rotation(pitch=-10, yaw=0, roll=0)  # Slight downward pitch for natural driving gaze
), attach_to=ego_vehicle)

# Import the display handler
from carla_display import CarlaDisplay

# Create display handler (runs pygame in background thread)
display = CarlaDisplay(width=W, height=H)
display.start()

# Set up camera callback to send images to display
camera.listen(display.on_image)

# Enable NPC autopilot
npc_vehicle.set_autopilot(True)
ego_vehicle.set_autopilot(True)

# Now you can continue executing cells - the display runs in the background!
# To stop the display, run: display.stop()

# Import the auto-respawn monitor (reload to pick up latest changes)
import importlib
import carla_respawn
importlib.reload(carla_respawn)
from carla_respawn import AutoRespawnMonitor

# Create and start auto-respawn monitor
# Respawns vehicles when either vehicle travels x += 200 from spawn, or after 30s, whichever comes first
monitor = AutoRespawnMonitor(
    world=world,
    ego_spawn_point=ego_spawn_point,
    npc_spawn_point=npc_spawn_point,
    ego_blueprint=ego_blueprint,
    npc_blueprint=npc_blueprint,
    camera_blueprint=cam_bp,
    camera_transform=carla.Transform(
        carla.Location(x=0.2, y=-0.36, z=1.2),
        carla.Rotation(pitch=-10, yaw=0, roll=0)
    ),
    display=display,
    spawn_x_threshold=200,  # x += 200 from spawn
    time_threshold=30  # 30 seconds
)

# Start monitoring with autopilot states (monitor will preserve these during respawn)
monitor.start(ego_vehicle, npc_vehicle, camera, 
              ego_autopilot=True, npc_autopilot=True)

# To stop monitoring later, run: monitor.stop()
# To change autopilot state: monitor.set_autopilot(ego=True, npc=False)

Pygame display started in background thread


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default


Auto-respawn monitor started


Respawn condition met: Position (ego_x=218.85, npc_x=221.87 >= 221.0)
Respawn triggered!
Vehicles respawned at x=21.0 (ego_autopilot=True, npc_autopilot=True)
Respawn condition met: Position (ego_x=222.13, npc_x=216.70 >= 221.0)
Respawn triggered!
Vehicles respawned at x=21.0 (ego_autopilot=True, npc_autopilot=True)


In [ ]:
# To stop the display, monitor, and camera:
if 'monitor' in globals():
    monitor.stop()
display.stop()
camera.stop()

# Get the list of actors
actors = world.get_actors()

# Delete all vehicles
for actor in actors:
    if actor.type_id.startswith('vehicle') or actor.type_id.startswith('sensor'):
        # print(actor.type_id)
        actor.destroy()
